# Naive Bayes & Bayesian Belief Networks

A complete guide: intuition, mathematics, and implementation from scratch.

## Table of Contents

1. [Intuition](#intuition)
2. [Bayes' Theorem](#bayes-theorem)
3. [The Naive Independence Assumption](#naive-assumption)
4. [Three Variants of Naive Bayes](#variants)
5. [Log-Probability Trick](#log-trick)
6. [Laplace Smoothing](#laplace)
7. [Setup & Imports](#imports)
8. [Gaussian Naive Bayes](#gaussian-nb)
9. [Multinomial Naive Bayes](#multinomial-nb)
10. [Bernoulli Naive Bayes](#bernoulli-nb)
11. [Complete NaiveBayes Class](#implementation)
12. [Demo: Classification (Iris — Gaussian)](#demo-gaussian)
13. [Demo: Text Classification (Multinomial)](#demo-text)
14. [Demo: Binary Features (Bernoulli)](#demo-bernoulli)
15. [Decision Boundary Visualization](#decision-boundary)
16. [When the Naive Assumption Fails](#when-fails)
17. [Bayesian Belief Networks: Intuition](#bbn-intuition)
18. [BBN: Formal Definition & Joint Factorization](#bbn-math)
19. [BBN: Conditional Independence & d-Separation](#bbn-dsep)
20. [BBN: Implementation from Scratch](#bbn-impl)
21. [BBN: Exact Inference by Enumeration](#bbn-inference)
22. [BBN: Classic Example — The Alarm Network](#bbn-alarm)
23. [Complexity Analysis](#complexity)
24. [Advantages and Disadvantages](#pros-cons)
25. [Summary](#summary)
26. [Sources](#sources)

<a id="intuition"></a>
## 1. Intuition

Naive Bayes is a **generative probabilistic classifier**. Instead of learning a decision boundary (like SVM or KNN), it models the probability of each class given the observed features using Bayes' theorem.

**Real-world analogy — spam filter:**

You receive an email containing the words *"free", "money", "click", "here"*. To decide if it is spam, you ask:

> *"Given that I've seen these words, what is the probability this email is spam?"*

You already know from past experience:
- 30% of all emails are spam ($P(\text{spam})$)
- The word *"free"* appears in 80% of spam but only 10% of non-spam ($P(\text{"free"} \mid \text{spam})$)
- Similarly for all other words

Naive Bayes combines all of these probabilities to produce a final verdict.

**Why "naive"?** The model assumes all features are **conditionally independent given the class**. Words in an email are obviously not independent — but this assumption, despite being wrong, produces surprisingly good classifiers in practice.

**Generative vs Discriminative:**

| | Generative (Naive Bayes) | Discriminative (SVM, Logistic Reg) |
|---|---|---|
| Models | $P(\mathbf{x}, C)$ jointly | $P(C \mid \mathbf{x})$ directly |
| Can generate new samples | Yes | No |
| Works well with small data | Yes | Less so |
| Sensitive to feature correlations | Yes (naive assumption) | No |

<a id="bayes-theorem"></a>
## 2. Bayes' Theorem

Bayes' theorem relates the **posterior** probability of class $C$ given features $\mathbf{x}$ to quantities we can estimate from data:

$$\underbrace{P(C \mid \mathbf{x})}_{\text{posterior}} = \frac{\overbrace{P(\mathbf{x} \mid C)}^{\text{likelihood}} \cdot \overbrace{P(C)}^{\text{prior}}}{\underbrace{P(\mathbf{x})}_{\text{evidence}}}$$

**Each term:**

| Term | Name | Meaning | How to estimate |
|---|---|---|---|
| $P(C)$ | **Prior** | How common is class $C$ overall? | Class frequencies in training set |
| $P(\mathbf{x} \mid C)$ | **Likelihood** | How probable are these features if the class is $C$? | Parametric model per feature per class |
| $P(\mathbf{x})$ | **Evidence** | How probable are these features overall? | Normalizing constant (same for all classes) |
| $P(C \mid \mathbf{x})$ | **Posterior** | What we want: how probable is class $C$ given what we observed? | Computed via Bayes' theorem |

**Classification rule** — pick the class with the highest posterior (MAP estimate):

$$\hat{C} = \underset{C}{\arg\max}\; P(C \mid \mathbf{x}) = \underset{C}{\arg\max}\; P(\mathbf{x} \mid C) \cdot P(C)$$

We can drop $P(\mathbf{x})$ because it is the same constant for all classes.

<a id="naive-assumption"></a>
## 3. The Naive Independence Assumption

For a feature vector $\mathbf{x} = (x_1, x_2, \ldots, x_p)$, computing the joint likelihood $P(\mathbf{x} \mid C)$ is generally intractable — it requires estimating a $p$-dimensional distribution.

Naive Bayes makes the **conditional independence assumption**:

$$P(\mathbf{x} \mid C) = P(x_1, x_2, \ldots, x_p \mid C) \overset{\text{naive}}{=} \prod_{j=1}^{p} P(x_j \mid C)$$

Each feature is modeled **independently** given the class. This reduces the problem from one $p$-dimensional distribution to $p$ one-dimensional distributions — easy to estimate even with little data.

**The full Naive Bayes classifier:**

$$\hat{C} = \underset{C}{\arg\max}\; P(C) \prod_{j=1}^{p} P(x_j \mid C)$$

**Is the assumption realistic?** Rarely. But the classifier still works well when:
- Features are only weakly correlated
- Data is scarce (fewer parameters = less overfitting)
- The task is text classification (bag of words)

> Domingos & Pazzani (1997) showed that Naive Bayes is optimal even under some degree of feature dependence, as long as the feature dependencies cancel out across classes.

<a id="variants"></a>
## 4. Three Variants of Naive Bayes

The three variants differ only in the model used for $P(x_j \mid C)$:

### 4.1 Gaussian Naive Bayes
For **continuous features**. Each feature per class is modeled as a Gaussian:

$$P(x_j \mid C = c) = \frac{1}{\sqrt{2\pi\,\sigma_{jc}^2}} \exp\!\left(-\frac{(x_j - \mu_{jc})^2}{2\sigma_{jc}^2}\right)$$

Parameters $\mu_{jc}$ and $\sigma_{jc}^2$ are estimated from training data by taking the mean and variance of feature $j$ among all samples of class $c$.

### 4.2 Multinomial Naive Bayes
For **count features** (e.g., word counts in text). Each feature is a count of occurrences:

$$P(\mathbf{x} \mid C = c) = \frac{(\sum_j x_j)!}{\prod_j x_j!} \prod_{j=1}^{p} \theta_{jc}^{x_j}$$

where $\theta_{jc} = P(\text{word}_j \mid C = c)$. Since the multinomial coefficient is constant across classes, we simplify to:

$$\hat{C} = \underset{c}{\arg\max}\; \log P(c) + \sum_{j=1}^{p} x_j \log \theta_{jc}$$

Parameters: $\theta_{jc} = \dfrac{\sum_{i: y_i=c} x_{ij} + \alpha}{\sum_{j'} \sum_{i: y_i=c} x_{ij'} + \alpha V}$ (with Laplace smoothing $\alpha$, $V$ = vocabulary size).

### 4.3 Bernoulli Naive Bayes
For **binary features** (e.g., word presence/absence). Each feature $x_j \in \{0, 1\}$:

$$P(x_j \mid C = c) = p_{jc}^{x_j} \cdot (1 - p_{jc})^{1 - x_j}$$

where $p_{jc} = P(x_j = 1 \mid C = c)$. Both presence **and absence** of a feature contribute.

Parameters: $p_{jc} = \dfrac{\sum_{i: y_i=c} x_{ij} + \alpha}{N_c + 2\alpha}$ (Laplace smoothing, $N_c$ = samples in class $c$).

### Summary

| Variant | Feature type | Likelihood model | Use case |
|---|---|---|---|
| Gaussian | Continuous | Gaussian distribution | Iris, sensor data |
| Multinomial | Counts / frequencies | Multinomial distribution | TF word counts, n-grams |
| Bernoulli | Binary (0/1) | Bernoulli distribution | Binary BOW, document classification |

<a id="log-trick"></a>
## 5. Log-Probability Trick

Multiplying many small probabilities causes **numerical underflow** — the product becomes so small that floating-point representation rounds it to zero.

**Solution:** take logarithms. Since $\log$ is monotone, $\arg\max$ is preserved:

$$\hat{C} = \underset{C}{\arg\max}\; \underbrace{\log P(C)}_{\text{log-prior}} + \sum_{j=1}^{p} \underbrace{\log P(x_j \mid C)}_{\text{log-likelihood}}$$

**Example — without log:**
$P(C) \cdot \prod_{j=1}^{100} P(x_j \mid C) \approx 0.5 \cdot (0.1)^{100} = 5 \times 10^{-101}$ → underflows to 0

**Example — with log:**
$\log(0.5) + 100 \cdot \log(0.1) \approx -0.693 - 230.3 = -230.9$ → numerically stable

All implementations in this notebook use log-probabilities internally.

<a id="laplace"></a>
## 6. Laplace Smoothing

**Problem:** if a word never appears in the training set for class $c$, then $P(\text{word} \mid C = c) = 0$, and the entire product collapses to zero — regardless of all other evidence.

**Solution — Laplace (additive) smoothing:** add a pseudocount $\alpha > 0$ to every count:

$$\hat{\theta}_{jc} = \frac{N_{jc} + \alpha}{N_c + \alpha V}$$

where $N_{jc}$ = count of feature $j$ in class $c$, $N_c$ = total count in class $c$, $V$ = vocabulary/feature size.

| $\alpha$ | Effect |
|---|---|
| $\alpha = 0$ | No smoothing — MLE estimate, zero-probability problem |
| $\alpha = 1$ | Laplace smoothing — classic default |
| $\alpha \to \infty$ | All features equally likely — pure prior |

$\alpha$ is a hyperparameter, typically set to 1 (Laplace) or tuned on a validation set.

<a id="imports"></a>
## 7. Setup & Imports

All core implementations use only NumPy. scikit-learn is imported for comparison and dataset loading only.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
from sklearn.datasets import load_iris, fetch_20newsgroups, make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
print("Imports OK")

<a id="gaussian-nb"></a>
## 8. Gaussian Naive Bayes

### Training

For each class $c$ and each feature $j$, estimate:

$$\mu_{jc} = \frac{1}{N_c} \sum_{i: y_i = c} x_{ij}, \qquad \sigma_{jc}^2 = \frac{1}{N_c} \sum_{i: y_i = c} (x_{ij} - \mu_{jc})^2$$

### Prediction

For query $\mathbf{x}$:

$$\log P(C = c \mid \mathbf{x}) \propto \log P(c) + \sum_{j=1}^{p} \log \mathcal{N}(x_j; \mu_{jc}, \sigma_{jc}^2)$$

$$= \log P(c) - \frac{1}{2} \sum_{j=1}^{p} \left[ \log(2\pi\sigma_{jc}^2) + \frac{(x_j - \mu_{jc})^2}{\sigma_{jc}^2} \right]$$

**Variance smoothing:** to avoid $\sigma_{jc}^2 = 0$ when all samples in a class share the same value for a feature, we add a small constant: $\sigma_{jc}^2 \leftarrow \sigma_{jc}^2 + \varepsilon$.

`GaussianNaiveBayes` computes $\mu_{jc}$ and $\sigma^2_{jc}$ for every (feature, class) pair at `fit` time using vectorized NumPy. `predict_log_proba` returns the unnormalized log-posterior for each class; `predict` returns the argmax.

In [ ]:
class GaussianNaiveBayes:
    """
    Gaussian Naive Bayes for continuous features.

    Parameters
    ----------
    var_smoothing : float — added to all variances to avoid division by zero
    """

    def __init__(self, var_smoothing=1e-9):
        self.var_smoothing = var_smoothing

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        self.classes_    = np.unique(y)
        n_samples        = len(y)
        self.log_priors_ = {}  # log P(C=c)
        self.means_      = {}  # mu_jc  shape (n_features,)
        self.vars_       = {}  # sigma^2_jc  shape (n_features,)

        for c in self.classes_:
            X_c = X[y == c]
            self.log_priors_[c] = np.log(len(X_c) / n_samples)
            self.means_[c]      = X_c.mean(axis=0)
            self.vars_[c]       = X_c.var(axis=0) + self.var_smoothing

        return self

    def predict_log_proba(self, X):
        """Return unnormalized log-posteriors, shape (n_samples, n_classes)."""
        X = np.asarray(X, dtype=float)
        log_posts = np.zeros((len(X), len(self.classes_)))
        for k, c in enumerate(self.classes_):
            mu  = self.means_[c]
            var = self.vars_[c]
            log_posts[:, k] = self.log_priors_[c] + (
                -0.5 * np.sum(np.log(2 * np.pi * var) + (X - mu) ** 2 / var, axis=1)
            )
        return log_posts

    def predict_proba(self, X):
        """Normalized class probabilities via log-sum-exp trick."""
        log_posts = self.predict_log_proba(X)
        log_posts -= log_posts.max(axis=1, keepdims=True)  # numerical stability
        probs = np.exp(log_posts)
        return probs / probs.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_log_proba(X), axis=1)]


print("GaussianNaiveBayes defined.")

<a id="multinomial-nb"></a>
## 9. Multinomial Naive Bayes

### Training

Estimate class-conditional word probabilities with Laplace smoothing:

$$\hat{\theta}_{jc} = \frac{\sum_{i: y_i = c} x_{ij} + \alpha}{\sum_{j'=1}^{V} \sum_{i: y_i = c} x_{ij'} + \alpha V}$$

### Prediction

$$\hat{C} = \underset{c}{\arg\max}\; \log P(c) + \sum_{j=1}^{V} x_j \log \hat{\theta}_{jc}$$

This is a **dot product** of the feature vector $\mathbf{x}$ with the log-probability vector $\log \boldsymbol{\theta}_c$ — efficient for sparse text data.

`MultinomialNaiveBayes` stores log-smoothed feature probabilities $\log \hat{\theta}_{jc}$ as a matrix of shape `(n_classes, n_features)`. Prediction becomes a matrix multiplication plus log-priors.

In [ ]:
class MultinomialNaiveBayes:
    """
    Multinomial Naive Bayes for count/frequency features.

    Parameters
    ----------
    alpha : float — Laplace smoothing parameter (default 1.0)
    """

    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        self.classes_    = np.unique(y)
        n_samples, n_features = X.shape

        # log P(C=c) for each class
        self.log_priors_ = np.array([
            np.log(np.sum(y == c) / n_samples) for c in self.classes_
        ])

        # log theta_jc: shape (n_classes, n_features)
        self.log_theta_ = np.zeros((len(self.classes_), n_features))
        for k, c in enumerate(self.classes_):
            X_c = X[y == c]
            counts = X_c.sum(axis=0) + self.alpha           # (n_features,)
            self.log_theta_[k] = np.log(counts / counts.sum())

        return self

    def predict_log_proba(self, X):
        """Log-posteriors via dot product: X @ log_theta.T + log_priors."""
        X = np.asarray(X, dtype=float)
        return X @ self.log_theta_.T + self.log_priors_  # (n_samples, n_classes)

    def predict_proba(self, X):
        log_posts = self.predict_log_proba(X)
        log_posts -= log_posts.max(axis=1, keepdims=True)
        probs = np.exp(log_posts)
        return probs / probs.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_log_proba(X), axis=1)]


print("MultinomialNaiveBayes defined.")

<a id="bernoulli-nb"></a>
## 10. Bernoulli Naive Bayes

### Training

Estimate the probability that feature $j$ is present (= 1) for class $c$:

$$\hat{p}_{jc} = \frac{\sum_{i: y_i = c} x_{ij} + \alpha}{N_c + 2\alpha}$$

### Prediction

$$\log P(C = c \mid \mathbf{x}) \propto \log P(c) + \sum_{j=1}^{p} \left[ x_j \log \hat{p}_{jc} + (1 - x_j) \log(1 - \hat{p}_{jc}) \right]$$

**Key difference from Multinomial:** absent features ($x_j = 0$) **actively penalize** classes that typically show that feature, via the $(1 - x_j) \log(1 - \hat{p}_{jc})$ term. Multinomial ignores absent features entirely.

`BernoulliNaiveBayes` stores $\log \hat{p}_{jc}$ and $\log(1 - \hat{p}_{jc})$ separately. Prediction uses a vectorized expression that handles both present and absent features simultaneously via broadcasting.

In [ ]:
class BernoulliNaiveBayes:
    """
    Bernoulli Naive Bayes for binary (0/1) features.

    Parameters
    ----------
    alpha : float — Laplace smoothing (default 1.0)
    """

    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        self.classes_    = np.unique(y)
        n_samples        = len(y)

        self.log_priors_ = np.array([
            np.log(np.sum(y == c) / n_samples) for c in self.classes_
        ])

        n_features = X.shape[1]
        # p_jc: (n_classes, n_features)
        self.log_p_          = np.zeros((len(self.classes_), n_features))
        self.log_complement_ = np.zeros((len(self.classes_), n_features))  # log(1 - p_jc)

        for k, c in enumerate(self.classes_):
            X_c = X[y == c]
            N_c = len(X_c)
            p_jc = (X_c.sum(axis=0) + self.alpha) / (N_c + 2 * self.alpha)
            self.log_p_[k]          = np.log(p_jc)
            self.log_complement_[k] = np.log(1 - p_jc)

        return self

    def predict_log_proba(self, X):
        X = np.asarray(X, dtype=float)
        # For each sample x: log P(c) + sum_j [x_j * log p_jc + (1-x_j) * log(1-p_jc)]
        # = log P(c) + x @ log_p.T + (1-x) @ log_complement.T
        pos = X @ self.log_p_.T                    # (n_samples, n_classes)
        neg = (1 - X) @ self.log_complement_.T     # (n_samples, n_classes)
        return self.log_priors_ + pos + neg

    def predict_proba(self, X):
        log_posts = self.predict_log_proba(X)
        log_posts -= log_posts.max(axis=1, keepdims=True)
        probs = np.exp(log_posts)
        return probs / probs.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_log_proba(X), axis=1)]


print("BernoulliNaiveBayes defined.")

<a id="implementation"></a>
## 11. Complete NaiveBayes Class

We wrap all three variants into a single sklearn-style class with a unified API, dispatching to the appropriate implementation based on the `distribution` parameter.

The `NaiveBayes` class is a thin dispatcher that instantiates the correct sub-model at `__init__` time and delegates all calls to it.

In [ ]:
class NaiveBayes:
    """
    Unified Naive Bayes classifier.

    Parameters
    ----------
    distribution : 'gaussian', 'multinomial', or 'bernoulli'
    alpha        : Laplace smoothing (Multinomial / Bernoulli only)
    var_smoothing: variance smoothing (Gaussian only)
    """

    _MODELS = {
        'gaussian':    GaussianNaiveBayes,
        'multinomial': MultinomialNaiveBayes,
        'bernoulli':   BernoulliNaiveBayes,
    }

    def __init__(self, distribution='gaussian', alpha=1.0, var_smoothing=1e-9):
        if distribution not in self._MODELS:
            raise ValueError(f"distribution must be one of {list(self._MODELS)}")
        self.distribution = distribution
        if distribution == 'gaussian':
            self._model = GaussianNaiveBayes(var_smoothing=var_smoothing)
        else:
            self._model = self._MODELS[distribution](alpha=alpha)

    def fit(self, X, y):                  return self._model.fit(X, y)
    def predict(self, X):                 return self._model.predict(X)
    def predict_proba(self, X):           return self._model.predict_proba(X)
    def predict_log_proba(self, X):       return self._model.predict_log_proba(X)


print("NaiveBayes wrapper defined.")

<a id="demo-gaussian"></a>
## 12. Demo: Classification (Iris — Gaussian)

Iris has 150 samples, 4 continuous features, 3 classes. Gaussian NB is the natural choice. We compare accuracy and per-class posteriors against sklearn.

Load Iris, split 80/20, fit Gaussian NB, compare accuracy and confusion matrix against sklearn's `GaussianNB`.

In [ ]:
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42)

# Our implementation
gnb = NaiveBayes(distribution='gaussian')
gnb.fit(X_tr, y_tr)
acc_ours = accuracy_score(y_te, gnb.predict(X_te))

# sklearn
sk_gnb = GaussianNB()
sk_gnb.fit(X_tr, y_tr)
acc_sk = accuracy_score(y_te, sk_gnb.predict(X_te))

print(f"Our GaussianNB accuracy: {acc_ours:.4f}")
print(f"sklearn  GaussianNB accuracy: {acc_sk:.4f}")

We plot the confusion matrix to see which classes are confused with each other, and display the learned Gaussian parameters (means per feature per class).

In [ ]:
cm = confusion_matrix(y_te, gnb.predict(X_te))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Confusion matrix
disp = ConfusionMatrixDisplay(cm, display_labels=iris.target_names)
disp.plot(ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix — Gaussian NB')

# Learned means per feature per class
means = np.array([gnb._model.means_[c] for c in gnb._model.classes_])
x = np.arange(len(iris.feature_names))
width = 0.25
for k, cls_name in enumerate(iris.target_names):
    axes[1].bar(x + k * width, means[k], width, label=cls_name)
axes[1].set_xticks(x + width)
axes[1].set_xticklabels(iris.feature_names, rotation=15, ha='right', fontsize=8)
axes[1].set_ylabel('Mean value')
axes[1].set_title('Learned μ per feature per class')
axes[1].legend()
plt.tight_layout()
plt.show()

<a id="demo-text"></a>
## 13. Demo: Text Classification (Multinomial)

The 20 Newsgroups dataset contains news articles from 20 categories. We select 4 categories, convert articles to word-count vectors (bag-of-words), and classify with Multinomial NB.

We load 4 newsgroup categories, build a vocabulary of the top 5,000 words using `CountVectorizer`, then train Multinomial NB on the count matrix.

In [ ]:
categories = ['sci.space', 'rec.sport.hockey', 'talk.politics.guns', 'comp.graphics']
news_train = fetch_20newsgroups(subset='train', categories=categories, remove=('headers', 'footers', 'quotes'))
news_test  = fetch_20newsgroups(subset='test',  categories=categories, remove=('headers', 'footers', 'quotes'))

vectorizer = CountVectorizer(max_features=5000, stop_words='english')
X_news_tr  = vectorizer.fit_transform(news_train.data).toarray()
X_news_te  = vectorizer.transform(news_test.data).toarray()
y_news_tr, y_news_te = news_train.target, news_test.target

print(f"Train: {X_news_tr.shape}  Test: {X_news_te.shape}")
print(f"Categories: {news_train.target_names}")

We train our Multinomial NB and sklearn's implementation, compare accuracy, and print the top 10 most discriminative words for each class.

In [ ]:
mnb = NaiveBayes(distribution='multinomial', alpha=1.0)
mnb.fit(X_news_tr, y_news_tr)
acc_ours = accuracy_score(y_news_te, mnb.predict(X_news_te))

sk_mnb = MultinomialNB(alpha=1.0)
sk_mnb.fit(X_news_tr, y_news_tr)
acc_sk = accuracy_score(y_news_te, sk_mnb.predict(X_news_te))

print(f"Our MultinomialNB accuracy:     {acc_ours:.4f}")
print(f"sklearn MultinomialNB accuracy: {acc_sk:.4f}")

# Top words per class
feature_names = np.array(vectorizer.get_feature_names_out())
log_theta = mnb._model.log_theta_   # (n_classes, n_features)

print("\nTop 10 most discriminative words per class:")
for k, cat in enumerate(categories):
    top_idx = np.argsort(log_theta[k])[-10:][::-1]
    print(f"  {cat:25s}: {', '.join(feature_names[top_idx])}")

<a id="demo-bernoulli"></a>
## 14. Demo: Binary Features (Bernoulli)

We convert the same text data to binary (word present/absent) and compare Bernoulli NB against Multinomial NB. Bernoulli typically performs better on short documents.

We binarize the count matrix (1 if word appears at least once, 0 otherwise), then train Bernoulli NB. We compare accuracy and discuss when Bernoulli outperforms Multinomial.

In [ ]:
X_bin_tr = (X_news_tr > 0).astype(float)
X_bin_te = (X_news_te > 0).astype(float)

bnb = NaiveBayes(distribution='bernoulli', alpha=1.0)
bnb.fit(X_bin_tr, y_news_tr)
acc_ours = accuracy_score(y_news_te, bnb.predict(X_bin_te))

sk_bnb = BernoulliNB(alpha=1.0)
sk_bnb.fit(X_bin_tr, y_news_tr)
acc_sk = accuracy_score(y_news_te, sk_bnb.predict(X_bin_te))

print(f"Our BernoulliNB accuracy:     {acc_ours:.4f}")
print(f"sklearn BernoulliNB accuracy: {acc_sk:.4f}")
print()
print("Summary — 4-class newsgroups:")
print(f"  Multinomial NB: {accuracy_score(y_news_te, mnb.predict(X_news_te)):.4f}")
print(f"  Bernoulli NB:   {acc_ours:.4f}")
print("(Multinomial usually wins on long documents with many word repetitions.")
print(" Bernoulli can win on short documents where frequency is less informative.)")

<a id="decision-boundary"></a>
## 15. Decision Boundary Visualization

Gaussian NB assumes per-class Gaussian distributions. Its decision boundaries are **quadratic** (ellipsoidal) in general. They become linear only when all classes share the same covariance matrix (this is the connection to Linear Discriminant Analysis).

We project Iris onto its first 2 features (sepal), train Gaussian NB, and plot the decision regions alongside the per-class Gaussian contours. We then compare to a 2D synthetic dataset to show the quadratic boundary.

In [ ]:
def plot_nb_boundary(nb, X, y, title='Gaussian NB Decision Boundary',
                     xlabel='x1', ylabel='x2', h=0.05):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                          np.arange(y_min, y_max, h))
    Z = nb.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    n_classes = len(np.unique(y))
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.contourf(xx, yy, Z, alpha=0.35, cmap=plt.cm.Pastel1,
                levels=np.arange(-0.5, n_classes + 0.5, 1))
    ax.contour(xx, yy, Z, colors='gray', linewidths=0.8, alpha=0.5)

    cmap = plt.cm.Set1
    for cls in np.unique(y):
        mask = y == cls
        ax.scatter(X[mask, 0], X[mask, 1],
                   c=[cmap(cls / n_classes)], edgecolors='k',
                   linewidth=0.5, s=50, label=f'Class {cls}')
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.set_title(title); ax.legend()
    plt.tight_layout()
    plt.show()


# Iris 2D
X_2d, y_2d = X_iris[:, :2], y_iris
gnb_2d = NaiveBayes(distribution='gaussian')
gnb_2d.fit(X_2d, y_2d)
plot_nb_boundary(gnb_2d, X_2d, y_2d,
                 title='Gaussian NB — Iris (sepal features)',
                 xlabel='Sepal Length', ylabel='Sepal Width')

# Synthetic 2D: two Gaussians with different covariances → quadratic boundary
np.random.seed(0)
X_q = np.vstack([
    np.random.randn(100, 2) * [1.0, 2.5],
    np.random.randn(100, 2) * [2.5, 1.0] + [4, 4]
])
y_q = np.array([0]*100 + [1]*100)
gnb_q = NaiveBayes(distribution='gaussian')
gnb_q.fit(X_q, y_q)
plot_nb_boundary(gnb_q, X_q, y_q,
                 title='Gaussian NB — Synthetic (quadratic boundary)',
                 xlabel='x1', ylabel='x2')

<a id="when-fails"></a>
## 16. When the Naive Assumption Fails

The naive independence assumption breaks when features are strongly correlated. Here we quantify the degradation on a synthetic dataset with increasing feature correlation.

We generate datasets with two correlated features and vary the Pearson correlation $\rho$ from 0 to 0.99. At each level we compare Gaussian NB accuracy against Logistic Regression (which is not affected by feature correlation in the same way).

In [ ]:
from sklearn.linear_model import LogisticRegression

rhos   = np.arange(0, 1.0, 0.1)
acc_nb, acc_lr = [], []

for rho in rhos:
    cov = [[1.0, rho], [rho, 1.0]]
    X_pos = np.random.multivariate_normal([1, 1], cov, 200)
    X_neg = np.random.multivariate_normal([-1, -1], cov, 200)
    X_c = np.vstack([X_pos, X_neg])
    y_c = np.array([1]*200 + [0]*200)

    Xtr, Xte, ytr, yte = train_test_split(X_c, y_c, test_size=0.3, random_state=0)

    nb = NaiveBayes(distribution='gaussian')
    nb.fit(Xtr, ytr)
    acc_nb.append(accuracy_score(yte, nb.predict(Xte)))

    lr = LogisticRegression(max_iter=500)
    lr.fit(Xtr, ytr)
    acc_lr.append(accuracy_score(yte, lr.predict(Xte)))

plt.figure(figsize=(8, 4))
plt.plot(rhos, acc_nb, 'b-o', label='Gaussian NB')
plt.plot(rhos, acc_lr, 'r-s', label='Logistic Regression')
plt.xlabel('Feature correlation ρ')
plt.ylabel('Test accuracy')
plt.title('Effect of Feature Correlation on Naive Bayes vs Logistic Regression')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<a id="bbn-intuition"></a>
## 17. Bayesian Belief Networks: Intuition

Naive Bayes assumes **all features are conditionally independent given the class**. This is convenient but often too restrictive.

A **Bayesian Belief Network (BBN)** — also called a Bayes Net or Bayesian Network — is a richer model that represents **arbitrary** conditional independence relationships among a set of random variables as a **Directed Acyclic Graph (DAG)**.

**Each node** = a random variable  
**Each directed edge** $X \to Y$ = $X$ is a direct cause/influence of $Y$

**Classic example — the Student network** (Russell & Norvig):

```
 Difficulty (D) ──────────────┐
                               ▼
     Intelligence (I) ───── Grade (G) ───── Letter (L)
                               ▼
                             SAT (S)
```

Reading the graph:
- Grade depends on both Difficulty and Intelligence
- Letter of recommendation depends only on Grade
- SAT score depends only on Intelligence
- Difficulty and Intelligence are independent (no edge, no common ancestor)

BBNs answer questions like:
- *"Given that a student got a good grade, what is the probability they are intelligent?"* (inference)
- *"If we observe the SAT score, does it change our belief about the grade?"* (reasoning under uncertainty)

**Naive Bayes as a special BBN:** Naive Bayes is exactly the BBN where the class node $C$ is the only parent of every feature node $x_j$, and no feature has any other parents or edges between features.

<a id="bbn-math"></a>
## 18. BBN: Formal Definition & Joint Factorization

### Definition

A BBN over variables $\mathbf{X} = \{X_1, \ldots, X_n\}$ is a pair $(G, P)$ where:
- $G$ is a **DAG** with nodes $X_1, \ldots, X_n$
- $P$ is a set of **Conditional Probability Tables (CPTs)**: one per node, $P(X_i \mid \text{Pa}(X_i))$, where $\text{Pa}(X_i)$ are the parent nodes of $X_i$ in $G$

### Joint Probability Factorization

The DAG encodes a specific factorization of the joint distribution via the **chain rule for Bayesian networks**:

$$P(X_1, X_2, \ldots, X_n) = \prod_{i=1}^{n} P(X_i \mid \text{Pa}(X_i))$$

This is the key equation: the joint distribution over $n$ variables is a product of $n$ local conditional distributions — one per node.

**Why this is powerful:** for $n$ binary variables, the full joint table has $2^n$ entries. The BBN factorization has at most $\sum_i 2^{|\text{Pa}(X_i)| + 1}$ **raw CPT entries** — exponentially smaller if the graph is sparse. The number of **free parameters** is half that, since each row of a binary CPT sums to 1 (one value is determined by the other).

**Example — Alarm network** (4 binary variables):

| Node | Parents | CPT size | Without BBN |
|---|---|---|---|
| Burglary $B$ | — | 2 | — |
| Earthquake $E$ | — | 2 | — |
| Alarm $A$ | $B, E$ | $2^3 = 8$ | — |
| JohnCalls $J$ | $A$ | $2^2 = 4$ | — |
| MaryCalls $M$ | $A$ | $2^2 = 4$ | — |
| **Total** | | **10 free parameters** (20 raw CPT entries) | $2^5 - 1 = 31$ |

The full joint factorizes as:

$$P(B, E, A, J, M) = P(B) \cdot P(E) \cdot P(A \mid B, E) \cdot P(J \mid A) \cdot P(M \mid A)$$

<a id="bbn-dsep"></a>
## 19. BBN: Conditional Independence & d-Separation

The graph encodes **conditional independence** statements via **d-separation** (directed separation). Two sets of nodes $\mathbf{X}$ and $\mathbf{Y}$ are d-separated by a set $\mathbf{Z}$ if every path between $\mathbf{X}$ and $\mathbf{Y}$ is **blocked** by $\mathbf{Z}$.

A path is blocked at node $V$ if:

| Pattern | Name | Blocked when |
|---|---|---|
| $X \to V \to Y$ | **Chain** | $V \in \mathbf{Z}$ (observing $V$ blocks the flow) |
| $X \leftarrow V \to Y$ | **Fork** | $V \in \mathbf{Z}$ (observing common cause blocks) |
| $X \to V \leftarrow Y$ | **Collider** | $V \notin \mathbf{Z}$ and no descendant of $V$ in $\mathbf{Z}$ |

If $\mathbf{X}$ and $\mathbf{Y}$ are d-separated by $\mathbf{Z}$, then:

$$\mathbf{X} \perp \mathbf{Y} \mid \mathbf{Z}$$

**Collider example — explaining away:**  
Burglary $B$ and Earthquake $E$ are marginally independent.  
But if we observe Alarm $A = \text{true}$ (the collider), they become **dependent**: knowing there was a burglary *reduces* the probability of an earthquake (and vice versa). This is called **explaining away** or Berkson's paradox.

<a id="bbn-impl"></a>
## 20. BBN: Implementation from Scratch

We implement a simple BBN for **discrete (binary/categorical) variables** with:
1. A DAG representation (adjacency dict)
2. CPT storage as nested dicts or NumPy arrays
3. Topological sort for correct evaluation order
4. Exact inference by full joint enumeration

The `BayesianNetwork` class stores the DAG and CPTs. `add_node` registers a variable with its CPT (a dict mapping parent-value tuples to probability distributions). `topological_sort` gives a valid evaluation order. `joint_probability` computes $P(\mathbf{x})$ for a complete assignment using the factorization formula.

In [ ]:
from itertools import product as cartesian_product
from collections import deque


class BayesianNetwork:
    """
    Discrete Bayesian Belief Network.

    Usage
    -----
    bn = BayesianNetwork()
    bn.add_node('B', parents=[], values=[True, False],
                cpt={(): {True: 0.001, False: 0.999}})
    ...
    p = bn.joint_probability({'B': True, 'E': False, ...})
    q = bn.query('B', evidence={'J': True, 'M': True})
    """

    def __init__(self):
        self.nodes   = {}   # node_name -> {'parents', 'values', 'cpt'}
        self.parents = {}   # node_name -> list of parent names
        self.children = defaultdict(list)

    def add_node(self, name, parents, values, cpt):
        """
        Register a node.

        Parameters
        ----------
        name    : str — variable name
        parents : list of str — names of parent nodes (ordered)
        values  : list — possible values of this variable
        cpt     : dict mapping parent-value tuple → {value: prob}
                  Use () as key for root nodes (no parents).
        """
        self.nodes[name]   = {'parents': parents, 'values': values, 'cpt': cpt}
        self.parents[name] = parents
        for p in parents:
            self.children[p].append(name)

    def topological_sort(self):
        """Kahn's algorithm for topological ordering."""
        in_degree = {n: len(v['parents']) for n, v in self.nodes.items()}
        queue     = deque(n for n, d in in_degree.items() if d == 0)
        order     = []
        while queue:
            node = queue.popleft()
            order.append(node)
            for child in self.children[node]:
                in_degree[child] -= 1
                if in_degree[child] == 0:
                    queue.append(child)
        return order

    def _p_node_given_parents(self, node, value, assignment):
        """P(node=value | parents=assignment) from CPT."""
        parent_vals = tuple(assignment[p] for p in self.parents[node])
        return self.nodes[node]['cpt'][parent_vals][value]

    def joint_probability(self, assignment):
        """
        Compute P(X1=x1, ..., Xn=xn) for a complete assignment.
        Uses the chain rule: P(X1,...,Xn) = Π P(Xi | Pa(Xi)).
        """
        prob = 1.0
        for node in self.nodes:
            prob *= self._p_node_given_parents(node, assignment[node], assignment)
        return prob

    def all_assignments(self, nodes=None):
        """Generator: yield all complete assignments over the specified (or all) nodes."""
        if nodes is None:
            nodes = list(self.nodes.keys())
        value_lists = [self.nodes[n]['values'] for n in nodes]
        for combo in cartesian_product(*value_lists):
            yield dict(zip(nodes, combo))


print("BayesianNetwork defined.")

<a id="bbn-inference"></a>
## 21. BBN: Exact Inference by Enumeration

### The Query

We want to compute the **posterior** probability of a query variable $Q$ given observed evidence $\mathbf{e}$:

$$P(Q = q \mid \mathbf{e}) = \frac{P(Q = q, \mathbf{e})}{P(\mathbf{e})} = \alpha \cdot P(Q = q, \mathbf{e})$$

where $\alpha = 1/P(\mathbf{e})$ is a normalizing constant.

### Inference by Enumeration

The simplest exact inference method: sum over all possible assignments of the **hidden variables** $\mathbf{H}$ (all variables that are neither $Q$ nor evidence):

$$P(Q = q \mid \mathbf{e}) = \alpha \sum_{\mathbf{h}} P(Q = q, \mathbf{e}, \mathbf{h})$$

We compute $P(Q = q, \mathbf{e}, \mathbf{h})$ using the joint factorization.

**Complexity:** $O(n \cdot 2^{|\mathbf{H}|})$ — exponential in the number of hidden variables. This is why more efficient algorithms (Variable Elimination, Belief Propagation) are used in practice for large networks.

We implement `query` as a method that enumerates over all assignments to hidden variables, sums up the joint probabilities for each value of the query variable, then normalizes. This is exact but exponential — suitable for small networks.

In [ ]:
def query(bn, query_var, evidence):
    """
    Exact inference by enumeration.

    P(query_var | evidence) via summing out hidden variables.

    Parameters
    ----------
    bn         : BayesianNetwork
    query_var  : str — name of the variable to query
    evidence   : dict — observed variable assignments

    Returns
    -------
    dict : {value: probability} for all values of query_var
    """
    hidden_vars = [n for n in bn.nodes if n != query_var and n not in evidence]
    q_values    = bn.nodes[query_var]['values']

    result = {q: 0.0 for q in q_values}

    if hidden_vars:
        hidden_value_lists = [bn.nodes[h]['values'] for h in hidden_vars]
        for hidden_combo in cartesian_product(*hidden_value_lists):
            hidden_assignment = dict(zip(hidden_vars, hidden_combo))
            for q_val in q_values:
                full = {**evidence, **hidden_assignment, query_var: q_val}
                result[q_val] += bn.joint_probability(full)
    else:
        for q_val in q_values:
            full = {**evidence, query_var: q_val}
            result[q_val] += bn.joint_probability(full)

    # Normalize
    total = sum(result.values())
    return {k: v / total for k, v in result.items()}


print("query() function defined.")

<a id="bbn-alarm"></a>
## 22. BBN: Classic Example — The Alarm Network

From Russell & Norvig, *Artificial Intelligence: A Modern Approach*, Chapter 13.

**Scenario:** There is an alarm at home. It can be triggered by a burglary or an earthquake. Your neighbors John and Mary call you if they hear the alarm.

```
  Burglary (B)    Earthquake (E)
        \              /
         \            /
          ▼          ▼
          Alarm (A)
         /        \
        ▼          ▼
  JohnCalls (J)  MaryCalls (M)
```

**Joint factorization:**

$$P(B, E, A, J, M) = P(B) \cdot P(E) \cdot P(A \mid B, E) \cdot P(J \mid A) \cdot P(M \mid A)$$

**CPT values** (from the textbook):

| $P(B = T)$ | $P(E = T)$ |
|---|---|
| 0.001 | 0.002 |

| $B$ | $E$ | $P(A = T \mid B, E)$ |
|---|---|---|
| T | T | 0.95 |
| T | F | 0.94 |
| F | T | 0.29 |
| F | F | 0.001 |

| $A$ | $P(J = T \mid A)$ | $P(M = T \mid A)$ |
|---|---|---|
| T | 0.90 | 0.70 |
| F | 0.05 | 0.01 |

We build the Alarm network by calling `add_node` for each variable, then run several inference queries to verify the results match the textbook values.

In [ ]:
bn = BayesianNetwork()

# Root nodes (no parents)
bn.add_node('B', parents=[], values=[True, False],
            cpt={(): {True: 0.001, False: 0.999}})

bn.add_node('E', parents=[], values=[True, False],
            cpt={(): {True: 0.002, False: 0.998}})

# Alarm depends on Burglary and Earthquake
bn.add_node('A', parents=['B', 'E'], values=[True, False],
            cpt={
                (True,  True):  {True: 0.95,  False: 0.05},
                (True,  False): {True: 0.94,  False: 0.06},
                (False, True):  {True: 0.29,  False: 0.71},
                (False, False): {True: 0.001, False: 0.999},
            })

# JohnCalls depends on Alarm
bn.add_node('J', parents=['A'], values=[True, False],
            cpt={
                (True,):  {True: 0.90, False: 0.10},
                (False,): {True: 0.05, False: 0.95},
            })

# MaryCalls depends on Alarm
bn.add_node('M', parents=['A'], values=[True, False],
            cpt={
                (True,):  {True: 0.70, False: 0.30},
                (False,): {True: 0.01, False: 0.99},
            })

print("Topological order:", bn.topological_sort())

# Verify a specific joint probability from the textbook
p_check = bn.joint_probability({'B': True, 'E': False, 'A': True, 'J': True, 'M': True})
print(f"P(B=T, E=F, A=T, J=T, M=T) = {p_check:.8f}  (textbook: ~0.000628)")

We run three classic queries from the textbook:
1. Prior probability of alarm
2. P(Burglary | John calls, Mary calls) — the canonical diagnostic query
3. Explaining away: how observing Alarm changes the independence of Burglary and Earthquake

In [ ]:
# Query 1: prior probability of alarm (no evidence)
p_alarm = query(bn, 'A', evidence={})
print(f"P(A=T)               = {p_alarm[True]:.6f}  (textbook: ~0.002516)")

# Query 2: P(Burglary | JohnCalls=T, MaryCalls=T)  — canonical example
p_burg = query(bn, 'B', evidence={'J': True, 'M': True})
print(f"P(B=T | J=T, M=T)    = {p_burg[True]:.6f}  (textbook: ~0.284)")

# Query 3: P(Alarm | JohnCalls=T, MaryCalls=T)
p_a_given_jm = query(bn, 'A', evidence={'J': True, 'M': True})
print(f"P(A=T | J=T, M=T)    = {p_a_given_jm[True]:.6f}  (textbook: ~0.761)")

# Query 4: Explaining away — P(B=T | A=T) vs P(B=T | A=T, E=T)
p_b_given_a    = query(bn, 'B', evidence={'A': True})
p_b_given_a_e  = query(bn, 'B', evidence={'A': True, 'E': True})
print()
print(f"Explaining away:")
print(f"  P(B=T | A=T)       = {p_b_given_a[True]:.6f}")
print(f"  P(B=T | A=T, E=T)  = {p_b_given_a_e[True]:.6f}")
print(f"  → Knowing earthquake occurred reduces probability of burglary")
print(f"    (earthquake 'explains away' the alarm — Berkson's paradox)")

We visualize how the posterior probability of Burglary changes as we add more evidence (prior → observe John calls → observe Mary also calls → observe Alarm is on).

In [ ]:
evidence_steps = [
    ({},                          'Prior P(B)'),
    ({'J': True},                 'P(B | J=T)'),
    ({'J': True, 'M': True},      'P(B | J=T, M=T)'),
    ({'J': True, 'M': True, 'A': True}, 'P(B | J=T, M=T, A=T)'),
]

probs = [query(bn, 'B', ev)[True] for ev, _ in evidence_steps]
labels = [lbl for _, lbl in evidence_steps]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(labels, probs, color=['steelblue', 'darkorange', 'green', 'crimson'])
ax.axhline(0.001, color='gray', linestyle='--', label='Prior P(B=T) = 0.001')
for bar, p in zip(bars, probs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{p:.4f}', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('P(Burglary = True)')
ax.set_title('Alarm Network: Posterior Probability of Burglary as Evidence Accumulates')
ax.set_ylim(0, 0.7)
ax.legend()
plt.tight_layout()
plt.show()

<a id="complexity"></a>
## 23. Complexity Analysis

### Naive Bayes

| Phase | Time | Space | Notes |
|---|---|---|---|
| **Training** | $O(np)$ | $O(Kp)$ | Compute mean/var (Gaussian) or counts (Multi/Bern) per class per feature |
| **Prediction** | $O(Kp)$ per sample | $O(Kp)$ | $K$ = classes, $p$ = features |

This is the **fastest classifier** to train and predict in the entire ML toolkit.

### Bayesian Belief Networks

| Operation | Exact inference | Notes |
|---|---|---|
| **Inference by enumeration** | $O(n \cdot d^n)$ | $d$ = domain size per variable |
| **Variable elimination** | $O(n \cdot d^{w+1})$ | $w$ = treewidth of the graph |
| **Belief propagation (trees)** | $O(n \cdot d^2)$ | Exact only on tree-structured graphs |
| **Sampling (MCMC)** | Approximate | Scales to large networks |

Exact inference is NP-hard in general (Dagum & Luby, 1993). Practical systems use approximate inference.

<a id="pros-cons"></a>
## 24. Advantages and Disadvantages

### Naive Bayes

| | Detail |
|---|---|
| **Fast** | $O(np)$ training — beats all other classifiers at scale |
| **Works with little data** | Fewer parameters → less overfitting |
| **Handles missing features** | Simply skip missing features in the product |
| **Online learning** | Update counts incrementally as new data arrives |
| **Calibrated probabilities** | Outputs actual probabilities, not just class labels |
| **Naive assumption** | Fails when features are strongly correlated |
| **Zero-frequency** | Unseen feature values → zero probability (fixed by Laplace) |

### Bayesian Belief Networks

| | Detail |
|---|---|
| **Encodes domain knowledge** | Graph structure can be hand-designed by experts |
| **Handles uncertainty** | Full probabilistic reasoning, not just point predictions |
| **Compact representation** | Fewer parameters than full joint when graph is sparse |
| **Interpretable** | Causal graph explains relationships visually |
| **Inference is hard** | Exact inference NP-hard; approximate inference required for large nets |
| **Structure learning is hard** | Learning the DAG from data is $O(2^n)$ in general |

<a id="summary"></a>
## 25. Summary

```
Naive Bayes
──────────────────────────────────────────────────────────
Core formula:
   P̂(C | x) ∝ P(C) · ∏ⱼ P(xⱼ | C)     [log version for stability]

Three variants:
  Gaussian    P(xⱼ|C) = N(μⱼc, σ²ⱼc)          → continuous features
  Multinomial P(xⱼ|C) ∝ θⱼc^xⱼ                → word counts
  Bernoulli   P(xⱼ|C) = pⱼc^xⱼ (1-pⱼc)^(1-xⱼ) → binary features

Training:   O(np)  — estimate class statistics per feature
Prediction: O(Kp)  — evaluate K log-likelihoods, take argmax
Laplace smoothing α prevents zero-probability problem

──────────────────────────────────────────────────────────
Bayesian Belief Networks
──────────────────────────────────────────────────────────
Core idea: represent joint P(X₁,...,Xₙ) as a DAG

Factorization:
  P(X₁,...,Xₙ) = ∏ᵢ P(Xᵢ | Pa(Xᵢ))

Naive Bayes = special BBN where class C is sole parent
              of every feature, no edges between features

Inference:
  P(Q | e) = α · Σₕ P(Q, e, h)   [enumeration — exact, exponential]

Key phenomena:
  d-separation → conditional independence reading from graph
  Explaining away (collider) → observing effect creates dependence
    between independent causes
```

<a id="sources"></a>
## 26. Sources

### Foundational Papers
- **Bayes, T. (1763)**. An Essay towards Solving a Problem in the Doctrine of Chances. *Philosophical Transactions of the Royal Society*, 53, 370–418. — Original Bayes' theorem.
- **Good, I.J. (1950)**. *Probability and the Weighing of Evidence*. Charles Griffin & Company. — Bayesian inference foundations.
- **Domingos, P. & Pazzani, M. (1997)**. On the Optimality of the Simple Bayesian Classifier under Zero-One Loss. *Machine Learning*, 29(2-3), 103–130. [DOI](https://doi.org/10.1023/A:1007413511361) — Why naive NB works despite violated assumptions.
- **Pearl, J. (1985)**. Bayesian Networks: A Model of Self-Activated Memory for Evidential Reasoning. *Proceedings of the 7th Annual Conference of the Cognitive Science Society*. — Original BBN paper.
- **Pearl, J. (1988)**. *Probabilistic Reasoning in Intelligent Systems*. Morgan Kaufmann. — The canonical BBN textbook.
- **Beinlich, I.A., Suermondt, H.J., Chavez, R.M. & Cooper, G.F. (1989)**. The ALARM Monitoring System: A Case Study with Two Probabilistic Inference Techniques for Belief Networks. *Proceedings of the 2nd European Conference on Artificial Intelligence in Medicine (AIME)*, pp. 247–256. — Original Alarm network.
- **Dagum, P. & Luby, M. (1993)**. Approximating probabilistic inference in Bayesian belief networks is NP-hard. *Artificial Intelligence*, 60(1), 141–153. — Hardness of exact BBN inference.

### Textbooks
- Russell, S. & Norvig, P. ***Artificial Intelligence: A Modern Approach*** (4th ed.). Chapters 12–13: Probabilistic Reasoning and BBNs. — Alarm network example from here.
- Bishop, C.M. ***Pattern Recognition and Machine Learning***. Chapter 8: Graphical Models.
- Murphy, K.P. ***Machine Learning: A Probabilistic Perspective***. Chapters 3, 10.

### Online Resources
- [scikit-learn — Naive Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html)
- [Wikipedia — Naive Bayes classifier](https://en.wikipedia.org/wiki/Naive_Bayes_classifier)
- [Wikipedia — Bayesian network](https://en.wikipedia.org/wiki/Bayesian_network)
- [Wikipedia — D-separation](https://en.wikipedia.org/wiki/Bayesian_network#d-separation)